# Particionado del manual en tiras.

In [ ]:
import json

# Cargar el manual desde el archivo
with open('../data/manual.json', 'r', encoding='utf-8') as f:
    manual = json.load(f)

# Extraer todos los artículos como chunks
chunks = []

for capitulo in manual['manual']['capitulos']:
    for articulo in capitulo['articulos']:
        chunk = {
            "chunk_id": f"{articulo['metadatos']['capitulo']}_{articulo['numero']}",
            "texto": articulo['contenido'],
            "metadatos": {
                "capitulo": articulo['metadatos']['capitulo'],
                "articulo": articulo['numero'],
                "tema_principal": articulo['metadatos']['tema_principal']
            }
        }
        chunks.append(chunk)

# Mostrar algunos ejemplos
for c in chunks[:3]:
    print("ID:", c["chunk_id"])
    print("Texto:", c["texto"][:200] + "...")  # Solo parte del texto
    print("Metadatos:", c["metadatos"])
    print("-" * 50)

## Generar Embeddings para cada chunk

In [9]:
import json
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Cargar los chunks (suponiendo que ya los generaste como en el paso anterior)
with open('../data/manual.json', 'r', encoding='utf-8') as f:
    manual = json.load(f)

# Extraer los chunks (igual que antes)
chunks = []
for capitulo in manual['manual']['capitulos']:
    for articulo in capitulo['articulos']:
        chunk = {
            "chunk_id": f"{articulo['metadatos']['capitulo']}_{articulo['numero']}",
            "texto": articulo['contenido'],
            "metadatos": {
                "capitulo": articulo['metadatos']['capitulo'],
                "articulo": articulo['numero'],
                "tema_principal": articulo['metadatos']['tema_principal']
            }
        }
        chunks.append(chunk)

# Limpiar texto opcionalmente
def limpiar_texto(texto):
    return ' '.join(texto.split())

for chunk in chunks:
    chunk["texto"] = limpiar_texto(chunk["texto"])

# Cargar modelo de embeddings
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Crear una lista con los textos
texts = [chunk["texto"] for chunk in chunks]

# Generar embeddings
embeddings = model.encode(texts, show_progress_bar=True)

# Convertir a numpy array (necesario para FAISS)
embeddings_np = np.array(embeddings).astype('float32')

# Guardar embeddings para reutilizarlos después (opcional)
np.save("embeddings_capitulo_I.npy", embeddings_np)

print("✅ Embeddings generados:", embeddings_np.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generados: (6, 384)


In [10]:
# Inicializar índice FAISS (L2 distance)
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)

# Agregar embeddings al índice
index.add(embeddings_np)

# Guardar el índice FAISS para usarlo después
faiss.write_index(index, "faiss_index_capitulo_I.index")

print("✅ Índice FAISS guardado")

✅ Índice FAISS guardado


In [ ]:
# Ejemplo de pregunta
pregunta = "¿Que beneficios puedo tener por participar en concursos?"

# Codificar pregunta
query_embedding = model.encode([pregunta]).astype('float32')

# Buscar los 3 artículos más similares
k = 3
distancias, indices = index.search(query_embedding, k)

# Mostrar resultados
print("🔍 Resultados de búsqueda:")
for i, idx in enumerate(indices[0]):
    print(f"\nResultado {i+1}:")
    print("ID:", chunks[idx]["chunk_id"])
    print("Texto:", chunks[idx]["texto"])
    print("Metadatos:", chunks[idx]["metadatos"])
    print("-" * 50)